# CS781 Capstone - Phase 7: Prior Case Retrieval (PCR)
Name: Aditya Kumar | Roll No: 240059 | Email: adityaj24@iitk.ac.in

In [11]:
from collections import Counter
import os, re, gc, glob, math, json, time, random, shutil, zipfile, hashlib, subprocess, sys, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(DEVICE, NUM_GPUS)

cuda 2


In [12]:
import os
PLAN = [
    ("lightweight", [dict(method="ia3"),   dict(method="kd")]),
]
EXP_TAG = "p7b"
SMOKE = False

hf_token = os.environ.get("HF_TOKEN")
HF_REPO, P7_DIR = "Exploration-Lab/CS-781-Capstone", "Phase_7_PCR"
LOCAL_DATA_DIR = None
INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
OUT_ROOT = os.environ.get("CS781_OUT_ROOT", "/kaggle/working")

CHUNK_WORDS, WINDOW = 200, 512
K_CHUNKS = 2
CHUNK_MODE = "headtail"
BIDIRECTIONAL = True
D_PROJ = 256

TEMP = 0.05
BATCH_Q, N_HARD, HARD_POOL, GRAD_ACCUM = 4, 3, 30, 2
EPOCHS = {"full": 3, "partial": 4, "lora": 4, "lorma": 4, "qlora": 4, "ia3": 5, "bitfit": 5, "kd": 4}
LR     = {"full": 5e-5, "partial": 1e-4, "lora": 1e-3, "lorma": 5e-4, "qlora": 1e-3, "ia3": 3e-3, "bitfit": 1e-3, "kd": 1e-4}
PROJ_LR, WEIGHT_DECAY, WARMUP_FRAC, PATIENCE = 2e-3, 0.01, 0.06, 3
GRAD_CKPT = True
LOG_EVERY = 1
TRAIN_QUERY_LIMIT = None
RUN_MAX_MIN = None
PEFT = dict(r=16, alpha=32, targets=("q_proj", "k_proj", "v_proj", "out_proj"), lorma_targets=("q_proj", "v_proj"), partial_last=4)
KD_T, KD_ALPHA, STUDENT_KEEP_EVERY = 2.0, 0.5, 2

TOPK = 7
HYBRID = True
FUSE_PER_RUN = True
ALPHA_GRID = [round(a, 2) for a in np.arange(0.3, 1.01, 0.1)]

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("plan:", [(a, [r["method"] for r in rs]) for a, rs in PLAN])

plan: [('lightweight', ['ia3', 'kd'])]


In [13]:
ROPE_BASE = 10000.0

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_seq_len:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

class MultiHeadAttention(nn.Module):
    """Generic multi-head attention. causal=True masks future query->key positions (self-attn
    in the decoder). query and key/value can come from different sequences (cross-attention)."""
    def __init__(self, d_model, n_heads, dropout=0.0, causal=False, max_seq_len=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.causal = causal
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
            self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x_q, x_kv):
        B, T, C = x_q.shape
        S = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if self.causal:
            scores = scores.masked_fill(~self.causal_mask[:T, :S], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

def feed_forward(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_tgt_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=True, max_seq_len=max_tgt_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x

class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff,
                 max_src_len, max_tgt_len, dropout=0.1):
        super().__init__()
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_src_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layers)])
        self.enc_ln_f = nn.LayerNorm(d_model)
        self.dec_ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode(self, src_ids):
        T = src_ids.shape[1]
        x = self.drop(self.token_emb(src_ids) + self.enc_pos[:T].unsqueeze(0))
        for block in self.encoder_blocks:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_input_ids, enc_out):
        T = tgt_input_ids.shape[1]
        x = self.drop(self.token_emb(tgt_input_ids) + self.dec_pos[:T].unsqueeze(0))
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.lm_head(self.dec_ln_f(x))

    def forward(self, src_ids, tgt_input_ids, tgt_target_ids=None):
        enc_out = self.encode(src_ids)
        logits = self.decode(tgt_input_ids, enc_out)
        loss = None
        if tgt_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_target_ids.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        enc_out = self.encode(src_ids[:, -self.max_src_len:])
        dec_ids = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_ids[:, -self.max_tgt_len:], enc_out)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=src_ids.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            dec_ids = torch.cat([dec_ids, next_token], dim=1)
        return dec_ids[:, 1:]

class PrefixAwareAttention(nn.Module):
    """Causal self-attention, except query positions inside the prefix (index < prefix_len)
    also get bidirectional access to the rest of the prefix. Positions >= prefix_len are
    plain causal. A single stack, single attention type per layer - neither a fully causal
    decoder nor a two-stack encoder-decoder."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, prefix_len):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        pos = torch.arange(T, device=x.device)
        in_prefix = pos < prefix_len
        prefix_bonus = in_prefix.unsqueeze(1) & in_prefix.unsqueeze(0)
        allowed = self.causal_mask[:T, :T] | prefix_bonus

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~allowed, float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class PrefixLMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixAwareAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x, prefix_len):
        x = x + self.attn(self.ln1(x), prefix_len)
        x = x + self.ff(self.ln2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([PrefixLMBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, prefix_len, targets=None, loss_mask=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x, prefix_len)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            per_tok = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), reduction="none").view(B, T)
            m = (torch.arange(T, device=idx.device) + 1 >= prefix_len).to(per_tok.dtype)
            loss = (per_tok * m).sum() / (m.sum() * B).clamp(min=1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, prefix_len, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond, prefix_len=min(prefix_len, idx_cond.shape[1]))
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class LightBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class LightweightLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LightBlock(d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight

def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class QKNormAttention(nn.Module):
    """Full multi-head causal attention with RoPE, plus QK-Norm: Q and K are each RMSNorm'd
    per head (over the head dimension) before the dot product, bounding attention logit scale
    independent of how large Q/K activations get during training - improves stability."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head)
        self.k_norm = RMSNorm(self.d_head)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class QKBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = QKNormAttention(d_model, n_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x

class RopeSwiGLUQKNormLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([QKBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

In [14]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

def find_phase1(arch):
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_model.pt"), recursive=True)
    assert hits, f"{arch}_model.pt not found under {INPUT_ROOT}. Attach the dataset made from your Phase-1 zips."
    d = os.path.dirname(hits[0])
    for n in (f"{arch}_bpe_tokenizer.json", "bpe_tokenizer.json"):
        if os.path.exists(os.path.join(d, n)): return hits[0], os.path.join(d, n)
    t = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_bpe_tokenizer.json"), recursive=True) or glob.glob(os.path.join(INPUT_ROOT, "**", "bpe_tokenizer.json"), recursive=True)
    assert t, "tokenizer json not found next to the model"
    return hits[0], t[0]

def load_phase1_backbone(arch, path):
    ck = torch.load(path, map_location="cpu", weights_only=False); cfg, sd = ck["config"], ck["state_dict"]; g = cfg.get
    if arch == "decoder": m = DecoderLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "encdec": m = EncoderDecoderLM(g("vocab_size"), g("d_model"), g("n_enc_layers"), g("n_dec_layers"), g("n_heads"), g("d_ff"), g("max_src_len"), g("max_tgt_len"), g("dropout", 0.1))
    elif arch == "hybrid": m = PrefixLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "lightweight": m = LightweightLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("n_kv_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "rope_qknorm": m = RopeSwiGLUQKNormLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    else: raise ValueError(arch)
    m.load_state_dict(sd, strict=True)
    return m, cfg, (g("max_seq_len") or g("max_src_len"))

def block_list_name(arch): return "encoder_blocks" if arch == "encdec" else "blocks"

def strip_unused(m, arch):
    m.lm_head = nn.Identity()
    if arch == "encdec": m.decoder_blocks = nn.ModuleList(); m.dec_ln_f = nn.Identity()

def prune_layers(m, arch, keep):
    blocks = getattr(m, block_list_name(arch)); setattr(m, block_list_name(arch), nn.ModuleList([blocks[i] for i in keep]))

In [15]:
class LoRALinear(nn.Module):
    """LoRA: y = Wx + s*B(Ax). rsLoRA = same with s = alpha/sqrt(r) instead of alpha/r (rank-stabilised)."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x): return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LoRMA+ : W' = (I + (alpha/r) B A) W0 (multiplicative); A=0, B=kaiming as in the authors' code, so W'=W0 at step 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.zeros(r, base.out_features)); self.B = nn.Parameter(torch.empty(base.out_features, r))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
    def forward(self, x):
        h = F.linear(x, self.base.weight); y = h + self.s * F.linear(F.linear(h, self.A), self.B)
        return y + self.base.bias if self.base.bias is not None else y

In [16]:
class BPE:
    """Phase-1 byte-level BPE (load + encode). Same merges and special ids as pretraining."""
    SPECIAL = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}; self.id_to_bytes = {i: bytes([i]) for i in range(256)}; self.special_to_id = {}
        for i, t in enumerate(self.SPECIAL): self.special_to_id[t] = 256 + i; self.id_to_bytes[256 + i] = t.encode()
    @staticmethod
    def _merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    @classmethod
    def load(cls, path):
        tok = cls()
        for ps, nid in json.load(open(path, encoding="utf-8"))["merges"].items():
            a, b = (int(x) for x in ps.split(",")); tok.merges[(a, b)] = nid; tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok
    def encode_piece(self, piece):
        seq = list(piece.encode("utf-8"))
        while len(seq) > 1:
            cand = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not cand: break
            pair = min(cand, key=lambda p: self.merges[p]); seq = self._merge(seq, pair, self.merges[pair])
        return seq

class WordTok:
    def __init__(self, bpe): self.bpe, self.pad, self.eosw, self.cache = bpe, bpe.special_to_id["<pad>"], bpe.special_to_id["</s>"], {}
    def encode_words(self, words, max_tok):
        ids = []
        for w in words:
            p = self.cache.get(w)
            if p is None: p = self.cache[w] = self.bpe.encode_piece(w) + [self.eosw]
            ids.extend(p)
        return np.array((ids or [self.eosw])[-max_tok:], dtype=np.int64)

def read_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)
def get_path(rel):
    if LOCAL_DATA_DIR: return os.path.join(LOCAL_DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_REPO, filename=f"{P7_DIR}/{rel}", repo_type="dataset", token=hf_token)

def _text(v):
    t = next((v[k] for k in ("text", "document", "case_text", "content") if k in v), None)
    assert t is not None, f"no text field (keys: {list(v)})"
    return " ".join(map(str, t)) if isinstance(t, list) else str(t)
def parse_queries(raw, labelled):
    pairs = list(raw.items()) if isinstance(raw, dict) and "id" not in raw else [(r.get("id", r.get("query_id")), r) for r in raw]
    out = []
    for qid, v in pairs:
        if not isinstance(v, dict): v = {"text": v}
        rel = None
        if labelled:
            k = next((k for k in ("relevant_candidates", "relevant", "relevant_ids", "labels") if k in v), None)
            assert k is not None, f"query {qid}: no relevant_candidates field (keys: {list(v)})"
            rel = [str(x) for x in v[k]]
        out.append(dict(id=str(qid), text=_text(v), rel=rel))
    return sorted(out, key=lambda d: d["id"])
def parse_pool(raw):
    pairs = list(raw.items()) if isinstance(raw, dict) and "id" not in raw else [(r.get("id", r.get("case_id", r.get("candidate_id"))), r) for r in raw]
    return sorted([dict(id=str(cid), text=_text(v if isinstance(v, dict) else {"text": v})) for cid, v in pairs], key=lambda d: d["id"])

SPLITS = {}
for sp, folder, lab in (("train", "Train", True), ("dev", "Dev", True), ("test", "Test", False)):
    q = parse_queries(read_json(get_path(f"{folder}/{sp}.json")), lab); c = parse_pool(read_json(get_path(f"{folder}/{sp}_candidates.json")))
    assert len({d["id"] for d in q}) == len(q) and len({d["id"] for d in c}) == len(c), f"duplicate ids in {sp}"
    SPLITS[sp] = dict(q=q, c=c)
if TRAIN_QUERY_LIMIT and len(SPLITS["train"]["q"]) > TRAIN_QUERY_LIMIT: SPLITS["train"]["q"] = random.Random(SEED).sample(SPLITS["train"]["q"], TRAIN_QUERY_LIMIT); SPLITS["train"]["q"].sort(key=lambda d: d["id"])
if SMOKE:
    for sp, n in (("train", 12), ("dev", 6), ("test", 4)):
        q = SPLITS[sp]["q"][:n]; keep = {r for d in q for r in (d["rel"] or [])}
        rest = [d for d in SPLITS[sp]["c"] if d["id"] not in keep][:60]; SPLITS[sp] = dict(q=q, c=sorted([d for d in SPLITS[sp]["c"] if d["id"] in keep] + rest, key=lambda d: d["id"]))
    EPOCHS = {k: 1 for k in EPOCHS}; print(">>> SMOKE MODE: tiny subset, 1 epoch <<<")
for sp, s in SPLITS.items():
    ids = {d["id"] for d in s["c"]}; s["cid"] = [d["id"] for d in s["c"]]; s["qid"] = [d["id"] for d in s["q"]]; s["cindex"] = {i: k for k, i in enumerate(s["cid"])}
    if s["q"][0]["rel"] is not None:
        s["gold"] = {d["id"]: {r for r in d["rel"] if r in ids} for d in s["q"]}
        miss = sum(len(d["rel"]) - len(s["gold"][d["id"]]) for d in s["q"])
        print(f"{sp:5s}: {len(s['q']):5d} queries | pool {len(s['c']):6d} cases | relevant/query {np.mean([len(g) for g in s['gold'].values()]):.2f} | queries without a relevant case in the pool: {sum(len(g)==0 for g in s['gold'].values())} | relevant ids missing from pool: {miss}")
    else: print(f"{sp:5s}: {len(s['q']):5d} queries | pool {len(s['c']):6d} cases")
print("words/case median:", int(np.median([len(d["text"].split()) for d in SPLITS["train"]["c"][:2000]])))

try:
    from sklearn.feature_extraction.text import TfidfVectorizer; HAVE_SK = True
except Exception: HAVE_SK = False; print("scikit-learn not found: hard negatives become random and the lexical fusion is disabled")
def lexical_blocks(q_texts, c_texts, step=256):
    vec = TfidfVectorizer(sublinear_tf=True, min_df=2 if len(c_texts) > 50 else 1, max_df=0.6, dtype=np.float32); C = vec.fit_transform(c_texts); Q = vec.transform(q_texts)
    for a in range(0, Q.shape[0], step): yield a, (Q[a:a + step] @ C.T).toarray()
LEX = {}
HARD, POS = {}, {}
tr = SPLITS["train"]
for i, d in enumerate(tr["q"]): POS[i] = [tr["cindex"][r] for r in d["rel"] if r in tr["cindex"]]
if HAVE_SK:
    for a, blk in lexical_blocks([d["text"] for d in tr["q"]], [d["text"] for d in tr["c"]]):
        for r in range(blk.shape[0]):
            i = a + r; row = blk[r].copy(); row[POS[i]] = -1e9
            if tr["q"][i]["id"] in tr["cindex"]: row[tr["cindex"][tr["q"][i]["id"]]] = -1e9
            HARD[i] = np.argpartition(-row, min(HARD_POOL, len(row) - 1))[:HARD_POOL].tolist()

def f1_at_k(pred, gold):
    tp = sum(len(set(pred[q][:TOPK]) & g) for q, g in gold.items()); n = sum(len(pred[q][:TOPK]) for q in gold)
    P, R = tp / max(n, 1), tp / max(sum(len(g) for g in gold.values()), 1)
    return (2 * P * R / (P + R) if P + R else 0.0), P, R
def top_ids(S, sp):
    s = SPLITS[sp]; S = S.copy()
    for i, qid in enumerate(s["qid"]):
        if qid in s["cindex"]: S[i, s["cindex"][qid]] = -1e9
    return {qid: [s["cid"][j] for j in np.argsort(-S[i])[:TOPK]] for i, qid in enumerate(s["qid"])}
def neutralize(S, sp):
    """a query's own cell (it is in the pool) is replaced by the row mean so it cannot distort z-scoring; top_ids() still excludes it."""
    s = SPLITS[sp]; S = np.array(S, dtype=np.float32); n = S.shape[1]
    for i, qid in enumerate(s["qid"]):
        j = s["cindex"].get(qid)
        if j is not None and n > 1: S[i, j] = 0.0; S[i, j] = S[i].sum() / (n - 1)
    return S
def zrow(S): return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)

def lex_variants(q_texts, c_texts):
    from sklearn.feature_extraction.text import CountVectorizer
    md = 2 if len(c_texts) > 50 else 1; out = {}
    cfgs = {"tf_uni": dict(sublinear_tf=True, min_df=md, max_df=0.6),
            "tf_stop": dict(sublinear_tf=True, min_df=md, max_df=0.5, stop_words="english"),
            "tf_bi": dict(sublinear_tf=True, min_df=md, max_df=0.5, stop_words="english", ngram_range=(1, 2), max_features=400000)}
    for n, kw in cfgs.items():
        vec = TfidfVectorizer(dtype=np.float32, **kw); Cm = vec.fit_transform(c_texts); Qm = vec.transform(q_texts); out[n] = (Qm @ Cm.T).toarray()
    cv = CountVectorizer(stop_words="english", min_df=md, max_df=0.7, dtype=np.float32); Cm = cv.fit_transform(c_texts).tocsr().astype(np.float32); Qm = cv.transform(q_texts).tocsr().astype(np.float32)
    N = Cm.shape[0]; df = np.asarray((Cm > 0).sum(0)).ravel(); idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32); dl = np.asarray(Cm.sum(1)).ravel(); k1, b = 1.2, 0.75
    rows = np.repeat(np.arange(N), np.diff(Cm.indptr)); tf = Cm.data; Cm.data = (tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / dl.mean()))) * idf[Cm.indices]
    Qm.data = np.log1p(Qm.data).astype(np.float32); out["bm25"] = (Qm @ Cm.T).toarray()
    return out

if HAVE_SK:
    LV = {sp: {k: neutralize(v, sp) for k, v in lex_variants([d["text"] for d in SPLITS[sp]["q"]], [d["text"] for d in SPLITS[sp]["c"]]).items()} for sp in ("dev", "test")}
    sc = {k: f1_at_k(top_ids(v, "dev"), SPLITS["dev"]["gold"]) for k, v in LV["dev"].items()}
    for k, v in sc.items(): print(f"  lexical {k:8s} dev F1@7 {v[0]:.4f} | P {v[1]:.4f} | R {v[2]:.4f}")
    order = sorted(sc, key=lambda k: -sc[k][0]); cand = {k: [k] for k in order}; cand["mix_all"] = order; cand["mix_top2"] = order[:2]; cand["mix_top3"] = order[:3]
    def mixed(sp, ks): return np.mean([zrow(LV[sp][k]) for k in ks], 0)
    res = {n: f1_at_k(top_ids(mixed("dev", ks), "dev"), SPLITS["dev"]["gold"])[0] for n, ks in cand.items()}
    for n, v in res.items():
        if n.startswith("mix"): print(f"  lexical {n:8s} dev F1@7 {v:.4f}")
    bestn = max(res, key=res.get); LEX["dev"], LEX["test"] = mixed("dev", cand[bestn]), mixed("test", cand[bestn])
    print(f"BASELINE (best lexical = {bestn}) dev F1@7 {res[bestn]:.4f}")

def pick_chunks(nc, K, mode):
    if nc <= K: return list(range(nc))
    if mode == "tail": return list(range(nc - K, nc))
    if mode == "head": return list(range(K))
    if mode == "headtail": h = K // 2; return list(range(h)) + list(range(nc - (K - h), nc))
    if mode == "spread": return sorted(set(np.linspace(0, nc - 1, K).round().astype(int).tolist()))
    raise ValueError(mode)
def pack_texts(texts, wt, mode, K):
    N = len(texts); X = np.full((N, K, WINDOW), wt.pad, np.int16); M = np.zeros((N, K), bool)
    for i, t in enumerate(texts):
        words = t.split(); nc = max(1, -(-len(words) // CHUNK_WORDS))
        for k, c in enumerate(pick_chunks(nc, K, mode)):
            ids = wt.encode_words(words[c * CHUNK_WORDS:(c + 1) * CHUNK_WORDS], WINDOW); X[i, k, :len(ids)] = ids; M[i, k] = True
    return torch.from_numpy(X), torch.from_numpy(M)

train:   827 queries | pool   4320 cases | relevant/query 6.42 | queries without a relevant case in the pool: 10 | relevant ids missing from pool: 0
dev  :   118 queries | pool   1023 cases | relevant/query 8.25 | queries without a relevant case in the pool: 0 | relevant ids missing from pool: 0
test :   237 queries | pool   1727 cases
words/case median: 3654
  lexical tf_uni   dev F1@7 0.3546 | P 0.3862 | R 0.3279
  lexical tf_stop  dev F1@7 0.3602 | P 0.3923 | R 0.3330
  lexical tf_bi    dev F1@7 0.4314 | P 0.4697 | R 0.3988
  lexical bm25     dev F1@7 0.1734 | P 0.1889 | R 0.1603
  lexical mix_all  dev F1@7 0.3813
  lexical mix_top2 dev F1@7 0.4136
  lexical mix_top3 dev F1@7 0.3969
BASELINE (best lexical = tf_bi) dev F1@7 0.4314


In [17]:
from torch.utils.checkpoint import checkpoint as torch_ckpt
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

class Encoder(nn.Module):
    """Phase-1 backbone -> mean-pool tokens per chunk -> mean over chunks -> centre/standardise (fixed, calibrated) -> linear projection -> L2-normalised case embedding."""
    def __init__(self, arch, backbone, d_model, pad_id, bidirectional, d_proj):
        super().__init__(); self.arch, self.backbone, self.bidir, self.pad_id, self.d_model = arch, backbone, bidirectional, pad_id, d_model
        self.proj = nn.Linear(d_model, d_proj)
        self.register_buffer("mu", torch.zeros(d_model)); self.register_buffer("sd", torch.ones(d_model))
        if bidirectional and arch in CAUSAL_ARCHS:
            for mod in backbone.modules():
                if hasattr(mod, "causal_mask"): mod.causal_mask.fill_(True)
    def run_block(self, b, x, *a):
        if GRAD_CKPT and self.training and torch.is_grad_enabled(): return torch_ckpt(b, x, *a, use_reentrant=False)
        return b(x, *a)
    def token_features(self, ids):
        m = self.backbone; e = m.token_emb(ids); T = e.shape[1]
        if self.arch == "decoder":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0))
            for b in m.blocks: x = self.run_block(b, x)
            return m.ln_f(x)
        if self.arch == "encdec":
            x = m.drop(e + m.enc_pos[:T].unsqueeze(0))
            for b in m.encoder_blocks: x = self.run_block(b, x)
            return m.enc_ln_f(x)
        if self.arch == "hybrid":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0)); plen = T if self.bidir else 0
            for b in m.blocks: x = self.run_block(b, x, plen)
            return m.ln_f(x)
        x = m.drop(e)
        for b in m.blocks: x = self.run_block(b, x, m.rope_cos, m.rope_sin)
        return m.norm_f(x)
    def pool(self, X, M):
        B, K, T = X.shape; ids = X.reshape(B * K, T).long(); valid = M.reshape(-1)
        h = self.token_features(ids[valid]).float(); tm = (ids[valid] != self.pad_id).float().unsqueeze(-1)
        vec = (h * tm).sum(1) / tm.sum(1).clamp(min=1)
        full = torch.zeros(B * K, self.d_model, device=vec.device, dtype=vec.dtype); full[valid] = vec; full = full.view(B, K, -1)
        w = M.float() / M.float().sum(1, keepdim=True).clamp(min=1); return (w.unsqueeze(-1) * full).sum(1)
    def forward(self, X, M):
        doc = (self.pool(X, M) - self.mu) / self.sd
        return F.normalize(self.proj(doc.to(self.proj.weight.dtype)).float(), dim=-1)

@torch.no_grad()
def calibrate(model, pk, n=192):
    """LM features share a huge common component (anisotropy). Estimate per-dimension mean/std on some training cases and remove them."""
    model.eval(); g = torch.Generator().manual_seed(0); idx = torch.randperm(len(pk["cX"]), generator=g)[:n]; outs = []
    for b in range(0, len(idx), 16):
        i = idx[b:b + 16]
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"): outs.append(model.pool(pk["cX"][i].to(DEVICE), pk["cM"][i].to(DEVICE)).float())
    V = torch.cat(outs); sd = V.std(0) if len(V) > 1 else torch.ones_like(V[0])
    model.mu.copy_(V.mean(0)); model.sd.copy_(sd.clamp(min=float(sd.median()) * 0.1 + 1e-6))

In [18]:
NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
       -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]

class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored as block-wise 4-bit NF4 (two codes per byte, block=64, absmax scale) + trainable LoRA."""
    def __init__(self, base, r, alpha, block=64):
        super().__init__()
        w = base.weight.detach().float().flatten(); self.n, self.shape, self.block = w.numel(), tuple(base.weight.shape), block
        if (-self.n) % block: w = F.pad(w, (0, (-self.n) % block))
        blocks = w.view(-1, block); absmax = blocks.abs().max(1, keepdim=True).values.clamp(min=1e-8)
        table = torch.tensor(NF4, device=w.device)
        codes = ((blocks / absmax).unsqueeze(-1) - table).abs().argmin(-1).to(torch.uint8)
        self.register_buffer("packed", (codes[:, 0::2] << 4) | codes[:, 1::2]); self.register_buffer("absmax", absmax.squeeze(1))
        self.register_buffer("table", table)
        self.register_buffer("bias", None if base.bias is None else base.bias.detach().clone())
        self.s = alpha / r; self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def dequant(self):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0xF).long()], -1).view(self.packed.shape[0], -1)
        return (self.table[codes] * self.absmax[:, None]).flatten()[:self.n].view(self.shape)
    def forward(self, x):
        return F.linear(x, self.dequant().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype)) + self.s * F.linear(F.linear(x, self.A), self.B)

class IA3Linear(nn.Module):
    """IA3: the output of a frozen linear layer is rescaled by a learned vector (init 1)."""
    def __init__(self, base): super().__init__(); self.base = base; self.vec = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.vec.to(x.dtype)
class IA3Scale(nn.Module):
    def __init__(self, d): super().__init__(); self.vec = nn.Parameter(torch.ones(d))
    def forward(self, x): return x * self.vec.to(x.dtype)

def _swap(root, names, make):
    n = 0
    for _, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(mod, cname, make(child)); n += 1
    return n

def apply_method(enc, method):
    bb, P = enc.backbone, PEFT
    if method in ("full", "kd"):
        for p in enc.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora": n = _swap(bb, P["targets"], lambda b: LoRALinear(b, P["r"], P["alpha"]))
        elif method == "lorma": n = _swap(bb, P["lorma_targets"], lambda b: LoRMALinear(b, P["r"], P["alpha"]))
        elif method == "qlora": n = _swap(bb, P["targets"], lambda b: QLoRALinear(b, P["r"], P["alpha"]))
        elif method == "ia3":
            n = _swap(bb, ("k_proj", "v_proj"), IA3Linear)
            for _, mod in list(bb.named_modules()):
                if type(mod).__name__ == "SwiGLU": mod.up_proj = IA3Linear(mod.up_proj); n += 1
                for cname, ch in list(mod.named_children()):
                    if cname == "ff" and isinstance(ch, nn.Sequential) and len(ch) == 4 and isinstance(ch[1], nn.GELU):
                        ch[1] = nn.Sequential(nn.GELU(), IA3Scale(ch[0].out_features)); n += 1
        elif method == "partial":
            n = 0
            for blk in list(getattr(bb, block_list_name(enc.arch)))[-P["partial_last"]:]:
                for p in blk.parameters(): p.requires_grad = True; n += 1
            for nm, p in bb.named_parameters():
                if nm.split(".")[0] in ("ln_f", "norm_f", "enc_ln_f"): p.requires_grad = True; n += 1
        elif method == "bitfit":
            n = 0
            for nm, p in bb.named_parameters():
                if nm.endswith(".bias"): p.requires_grad = True; n += 1
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                    for p in mod.parameters(): p.requires_grad = True; n += 1
        else: raise ValueError(f"unknown method {method}")
        assert n > 0, f"{method}: nothing to adapt"
    for p in enc.proj.parameters(): p.requires_grad = True
    tr = sum(p.numel() for p in enc.parameters() if p.requires_grad); tot = sum(p.numel() for p in enc.parameters())
    print(f"[{method}] trainable {tr/1e6:.3f}M of {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad} | {"mu", "sd"}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

In [19]:
def build_enc(arch, p1_path, method, keep_every=None):
    bb, cfg, max_len = load_phase1_backbone(arch, p1_path); strip_unused(bb, arch)
    if keep_every: prune_layers(bb, arch, list(range(0, len(getattr(bb, block_list_name(arch))), keep_every)))
    assert WINDOW <= max_len, f"WINDOW {WINDOW} exceeds the Phase-1 max length {max_len}"
    enc = Encoder(arch, bb, cfg["d_model"], WT.pad, BIDIRECTIONAL, D_PROJ); apply_method(enc, method)
    return enc.to(DEVICE)

@torch.no_grad()
def embed(model, X, M, bs=16):
    model.eval(); out = []
    for b in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"): out.append(model(X[b:b + bs].to(DEVICE), M[b:b + bs].to(DEVICE)).float().cpu())
    return torch.cat(out)

def dense_scores(model, pk, sp): return neutralize((embed(model, pk["qX"], pk["qM"]) @ embed(model, pk["cX"], pk["cM"]).T).numpy(), sp)

def fuse(dense, lex, a): return dense if (lex is None or a >= 1.0) else a * zrow(dense) + (1 - a) * zrow(lex)
def tune_alpha(dense, lex, sp):
    if not (HYBRID and lex is not None): return 1.0, f1_at_k(top_ids(dense, sp), SPLITS[sp]["gold"])[0]
    best = max(((f1_at_k(top_ids(fuse(dense, lex, a), sp), SPLITS[sp]["gold"])[0], a) for a in ALPHA_GRID), key=lambda t: (t[0], t[1]))
    return best[1], best[0]

def make_batch(pk, qis, rng):
    docs, qrows = [], []
    for qi in qis:
        p = rng.choice(POS[qi]); gold = set(POS[qi]); negs = rng.sample(HARD[qi], min(N_HARD, len(HARD[qi]))) if HARD.get(qi) else []
        while len(negs) < N_HARD:
            j = rng.randrange(len(tr["c"]))
            if j not in gold and j not in negs: negs.append(j)
        docs += [p] + negs; qrows.append(qi)
    return qrows, docs

def fit(model, pk, dev_pk, ckdir, method, teacher=None, seed=0):
    os.makedirs(ckdir, exist_ok=True); last_p, best_p = f"{ckdir}/last.pt", f"{ckdir}/best.pt"
    proj_ids = {id(p) for p in model.proj.parameters()}; trn = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": [p for p in trn if id(p) in proj_ids], "lr": PROJ_LR}, {"params": [p for p in trn if id(p) not in proj_ids], "lr": LR[method]}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY); base = [g["lr"] for g in opt.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda"); rng = random.Random(seed)
    train_q = [i for i in range(len(tr["q"])) if POS[i]]; assert train_q, "no training query has a relevant candidate in its pool"
    spe = max(len(train_q) // (BATCH_Q * GRAD_ACCUM), 1); epochs = EPOCHS[method]; total, warm = epochs * spe, max(int(WARMUP_FRAC * epochs * spe), 1)
    start, best, hist, bad = 0, -1.0, [], 0
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False); model.load_state_dict(ck["trainable"], strict=False)
        opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); start, best, hist, bad = ck["epoch"], ck["best"], ck["hist"], ck["bad"]; print(f"resumed at epoch {start}/{epochs}")
    step, t0 = start * spe, time.time()
    for ep in range(start, epochs):
        model.train(); order = train_q[:]; rng.shuffle(order); run = 0.0; te = time.time(); opt.zero_grad(set_to_none=True)
        for k in range(1, spe + 1):
            sl = 0.0
            for a in range(GRAD_ACCUM):
                qis = order[((k - 1) * GRAD_ACCUM + a) * BATCH_Q: ((k - 1) * GRAD_ACCUM + a + 1) * BATCH_Q] or order[:BATCH_Q]
                qrows, docs = make_batch(pk, qis, rng); H1 = 1 + N_HARD
                qX, qM = pk["qX"][qrows].to(DEVICE), pk["qM"][qrows].to(DEVICE); dX, dM = pk["cX"][docs].to(DEVICE), pk["cM"][docs].to(DEVICE)
                with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                    qe, de = model(qX, qM), model(dX, dM); logits = (qe.float() @ de.float().T) / TEMP
                    tgt = torch.arange(len(qis), device=DEVICE) * H1; loss = F.cross_entropy(logits, tgt)
                    if teacher is not None:
                        with torch.no_grad(): tl = (teacher(qX, qM).float() @ teacher(dX, dM).float().T) / TEMP
                        kd = F.kl_div(F.log_softmax(logits / KD_T, -1), F.softmax(tl / KD_T, -1), reduction="batchmean") * KD_T * KD_T
                        loss = KD_ALPHA * loss + (1 - KD_ALPHA) * kd
                    loss = loss / GRAD_ACCUM
                scaler.scale(loss).backward(); sl += loss.item() * GRAD_ACCUM
            f = (step + 1) / warm if step < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))
            for g, b in zip(opt.param_groups, base): g["lr"] = b * f
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trn, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1; run += sl / GRAD_ACCUM
            if LOG_EVERY and step % LOG_EVERY == 0:
                el = time.time() - te; print(f"    ep {ep+1}/{epochs} step {k}/{spe} | loss {sl / GRAD_ACCUM:.4f} | lr {LR[method] * f:.2e} | {el:.0f}s elapsed, epoch ETA {el / k * (spe - k):.0f}s", flush=True)
        f1 = f1_at_k(top_ids(dense_scores(model, dev_pk, "dev"), "dev"), SPLITS["dev"]["gold"])[0]; hist.append(dict(epoch=ep + 1, loss=run / spe, dev_f1_dense=f1))
        print(f"  epoch {ep+1}/{epochs} | loss {run/spe:.4f} | dev dense F1@7 {f1:.4f} | {(time.time()-t0)/60:.1f} min", flush=True)
        if f1 > best: best, bad = f1, 0; torch.save({"trainable": trainable_state(model)}, best_p + ".tmp"); os.replace(best_p + ".tmp", best_p)
        else: bad += 1
        torch.save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": ep + 1, "best": best, "hist": hist, "bad": bad}, last_p + ".tmp"); os.replace(last_p + ".tmp", last_p)
        if bad >= PATIENCE: print("  early stop (patience)"); break
        if RUN_MAX_MIN and (time.time() - t0) / 60 > RUN_MAX_MIN: print("  stopping: time limit for this run"); break
    model.load_state_dict(torch.load(best_p, map_location="cpu", weights_only=False)["trainable"], strict=False)
    return hist, best

In [20]:
def setup_arch(arch):
    global ARCH, WORK, P1_MODEL, WT, TOK_SHA, PACKS
    ARCH = arch; WORK = os.path.join(OUT_ROOT, f"{EXP_TAG}_{arch}"); os.makedirs(WORK, exist_ok=True); PACKS = {}
    P1_MODEL, p1_tok = find_phase1(arch); WT = WordTok(BPE.load(p1_tok))
    TOK_SHA = hashlib.sha256(open(p1_tok, "rb").read()).hexdigest(); print(f"\n##### {arch}: model {P1_MODEL} | tokenizer {TOK_SHA[:10]}")
def get_packs(mode, K):
    if (mode, K) not in PACKS:
        pk = {}
        for sp, s in SPLITS.items():
            qX, qM = pack_texts([d["text"] for d in s["q"]], WT, mode, K); cX, cM = pack_texts([d["text"] for d in s["c"]], WT, mode, K); pk[sp] = dict(qX=qX, qM=qM, cX=cX, cM=cM)
        PACKS[(mode, K)] = pk
    return PACKS[(mode, K)]

def write_submission(S, name, outdir):
    sub = top_ids(S, "test"); assert all(len(set(v)) == TOPK and len(v) == TOPK for v in sub.values()), "need exactly 7 unique candidates per query (pool too small?)"
    pj = f"{outdir}/prediction.json"; json.dump(sub, open(pj, "w")); zp = f"{OUT_ROOT}/{name}_submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return zp

def run_one(spec):
    method = spec["method"]; name = spec.get("name", method); mode = spec.get("mode", CHUNK_MODE); K = spec.get("k", K_CHUNKS); outdir = f"{WORK}/{name}"; os.makedirs(outdir, exist_ok=True)
    if os.path.exists(f"{outdir}/done.json"):
        print(f"[{ARCH}/{name}] already done -> skipped"); zp = f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_{name}_submission.zip"
        if not os.path.exists(zp) and os.path.exists(f"{outdir}/prediction.json"):
            with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{outdir}/prediction.json", arcname="prediction.json")
        return json.load(open(f"{outdir}/done.json"))
    print(f"\n{'='*20} {ARCH} / {name}  (chunks={mode}, K={K}) {'='*20}"); t_start = time.time()
    model = build_enc(ARCH, P1_MODEL, method, STUDENT_KEEP_EVERY if method == "kd" else None); pk = get_packs(mode, K); teacher = None; calibrate(model, pk["train"])
    if method == "kd":
        cands = [m for m in (json.load(open(p)) for p in glob.glob(f"{WORK}/*/done.json")) if m["method"] != "kd" and m["mode"] == mode and m["k"] == K]
        assert cands, "kd needs a finished non-kd run of this model (same chunk mode/K) earlier in its list"
        tm = max(cands, key=lambda m: m["dev_f1_dense"]); print(f"  teacher = {tm['name']} (dev dense F1@7 {tm['dev_f1_dense']:.4f}); student keeps every {STUDENT_KEEP_EVERY}nd block")
        teacher = build_enc(ARCH, P1_MODEL, tm["method"]); teacher.load_state_dict(torch.load(f"{WORK}/{tm['name']}/ckpt/best.pt", map_location="cpu", weights_only=False)["trainable"], strict=False)
        teacher.eval(); [p.requires_grad_(False) for p in teacher.parameters()]
    hist, best = fit(model, pk["train"], pk["dev"], f"{outdir}/ckpt", method, teacher)
    t_e = time.time(); Sd = dense_scores(model, pk["dev"], "dev"); n_docs = len(pk["dev"]["cX"]) + len(pk["dev"]["qX"]); docs_per_s = n_docs / max(time.time() - t_e, 1e-6); St = dense_scores(model, pk["test"], "test")
    ld, lt = LEX.get("dev"), LEX.get("test"); a, hy = tune_alpha(Sd, ld, "dev"); dn = f1_at_k(top_ids(Sd, "dev"), SPLITS["dev"]["gold"]); hyb = f1_at_k(top_ids(fuse(Sd, ld, a), "dev"), SPLITS["dev"]["gold"])
    np.save(f"{outdir}/scores_dev.npy", Sd.astype(np.float16)); np.save(f"{outdir}/scores_test.npy", St.astype(np.float16))
    meta = dict(task="phase7_pcr", name=name, arch=ARCH, method=method, mode=mode, k=K, tag=EXP_TAG, tokenizer_sha=TOK_SHA, dev_f1_dense=dn[0], dev_p_dense=dn[1], dev_r_dense=dn[2],
                alpha=a, dev_f1_hybrid=hyb[0], dev_p_hybrid=hyb[1], dev_r_hybrid=hyb[2], dev_q_ids=SPLITS["dev"]["qid"], dev_c_ids=SPLITS["dev"]["cid"], test_q_ids=SPLITS["test"]["qid"], test_c_ids=SPLITS["test"]["cid"],
                trainable_M=round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3), total_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 2), embed_dim=D_PROJ,
                encode_docs_per_s=round(docs_per_s, 1), lr=LR[method], epochs_run=len(hist), minutes=round((time.time() - t_start) / 60, 1), smoke=SMOKE)
    json.dump(meta, open(f"{outdir}/meta.json", "w"), indent=1); zp = write_submission(fuse(St, lt, a if FUSE_PER_RUN else 1.0), f"{EXP_TAG}_{ARCH}_{name}", outdir)
    print(f"  dev: dense F1@7 {dn[0]:.4f} (P {dn[1]:.3f} R {dn[2]:.3f}) | fused with TF-IDF (alpha={a}) F1@7 {hyb[0]:.4f} | {docs_per_s:.0f} docs/s encode\n  submitted ranking: {'dense + TF-IDF' if FUSE_PER_RUN else 'dense Phase-1 score only'}\n  submission: {zp}")
    json.dump(meta, open(f"{outdir}/done.json", "w")); del model, teacher; gc.collect(); torch.cuda.empty_cache()
    return meta

def zip_dir(zpath, files):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f, arc in files:
            if os.path.exists(f): z.write(f, arc)
    print(f"{os.path.getsize(zpath)/1e6:.1f} MB  {zpath}")

RESULTS = []
for arch, runs in PLAN:
    setup_arch(arch)
    for s in runs: RESULTS.append(run_one(s))
    done = [os.path.dirname(p) for p in glob.glob(f"{WORK}/*/done.json")]
    zip_dir(f"{OUT_ROOT}/{EXP_TAG}_{arch}_bundle.zip", [(f"{d}/{n}", f"{os.path.basename(WORK)}/{os.path.basename(d)}/{n}") for d in done for n in ("scores_dev.npy", "scores_test.npy", "meta.json")])
print("\nSESSION SUMMARY"); [print(f"  {m['arch']:11s} {m['name']:8s} dense F1@7 {m['dev_f1_dense']:.4f} | fused {m['dev_f1_hybrid']:.4f} | {m['minutes']} min") for m in RESULTS]
dirs = [os.path.join(OUT_ROOT, d) for d in os.listdir(OUT_ROOT) if d.startswith(f"{EXP_TAG}_") and os.path.isdir(os.path.join(OUT_ROOT, d))]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_ALL_outputs.zip", [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT_ROOT)) for d in dirs for dp, _, fs in os.walk(d) for f in fs if f != "last.pt"])
print("Download the *_submission.zip files and the ALL_outputs zip from the Output panel BEFORE closing. Do NOT use Save Version / Save & Run All.")


##### lightweight: model /kaggle/input/datasets/radynx/lightweight/lightweight_model.pt | tokenizer 8623378cb3

==================== lightweight / ia3  (chunks=headtail, K=2) ====================
[ia3] trainable 0.238M of 110.14M (0.22%)
    ep 1/5 step 1/102 | loss 6.3904 | lr 1.00e-04 | 6s elapsed, epoch ETA 616s
    ep 1/5 step 2/102 | loss 6.9225 | lr 2.00e-04 | 12s elapsed, epoch ETA 618s
    ep 1/5 step 3/102 | loss 11.9676 | lr 3.00e-04 | 19s elapsed, epoch ETA 625s
    ep 1/5 step 4/102 | loss 10.0839 | lr 4.00e-04 | 26s elapsed, epoch ETA 633s
    ep 1/5 step 5/102 | loss 4.6931 | lr 5.00e-04 | 33s elapsed, epoch ETA 639s
    ep 1/5 step 6/102 | loss 6.5663 | lr 6.00e-04 | 40s elapsed, epoch ETA 636s
    ep 1/5 step 7/102 | loss 10.0291 | lr 7.00e-04 | 46s elapsed, epoch ETA 628s
    ep 1/5 step 8/102 | loss 5.4854 | lr 8.00e-04 | 53s elapsed, epoch ETA 617s
    ep 1/5 step 9/102 | loss 4.1938 | lr 9.00e-04 | 59s elapsed, epoch ETA 607s
    ep 1/5 step 10/102 | loss 6.2649 | 

In [21]:
ENSEMBLE = True
if ENSEMBLE:
    metas = []
    for root in (OUT_ROOT, INPUT_ROOT):
        for mp in glob.glob(os.path.join(root, "**", "meta.json"), recursive=True):
            try: m = json.load(open(mp))
            except Exception: continue
            if m.get("task") == "phase7_pcr" and not m.get("smoke") and os.path.exists(os.path.join(os.path.dirname(mp), "scores_dev.npy")):
                m["_dir"] = os.path.dirname(mp); metas.append(m)
    seen = {}
    for m in metas: seen.setdefault((m["arch"], m["name"]), m)
    metas = sorted(seen.values(), key=lambda m: -m["dev_f1_dense"]); assert metas, "no finished runs found"
    metas = [m for m in metas if m["dev_c_ids"] == SPLITS["dev"]["cid"] and m["dev_q_ids"] == SPLITS["dev"]["qid"] and m["test_c_ids"] == SPLITS["test"]["cid"]]
    SD = [zrow(np.load(m["_dir"] + "/scores_dev.npy").astype(np.float32)) for m in metas]; ST = [zrow(np.load(m["_dir"] + "/scores_test.npy").astype(np.float32)) for m in metas]
    print(f"{len(metas)} runs found:"); [print(f"  {m['arch']:11s} {m['name']:8s} dev dense F1@7 {m['dev_f1_dense']:.4f} | fused {m['dev_f1_hybrid']:.4f}") for m in metas]
    def finish(idx, name):
        sd, st = np.mean([SD[i] for i in idx], 0), np.mean([ST[i] for i in idx], 0); ld, lt = LEX.get("dev"), LEX.get("test"); a, hy = tune_alpha(sd, ld, "dev")
        dn = f1_at_k(top_ids(sd, "dev"), SPLITS["dev"]["gold"]); od = f"{OUT_ROOT}/{EXP_TAG}_ensemble_{name}"; os.makedirs(od, exist_ok=True)
        zp = write_submission(st, f"{EXP_TAG}_ensemble_{name}", od)
        print(f"  {name:6s} members={len(idx)} dev dense F1@7 {dn[0]:.4f} -> {zp}")
        if HYBRID and lt is not None:
            odf = od + "_fused"; os.makedirs(odf, exist_ok=True); zf = write_submission(fuse(st, lt, a), f"{EXP_TAG}_ensemble_{name}_fused", odf)
            print(f"         fused (alpha={a}) dev F1@7 {hy:.4f} -> {zf}")
    finish(list(range(len(metas))), "all"); finish(list(range(min(3, len(metas)))), "top3")

2 runs found:
  lightweight kd       dev dense F1@7 0.1078 | fused 0.4425
  lightweight ia3      dev dense F1@7 0.1023 | fused 0.4358
  all    members=2 dev dense F1@7 0.1234 -> /kaggle/working/p7b_ensemble_all_submission.zip
         fused (alpha=0.4) dev F1@7 0.4414 -> /kaggle/working/p7b_ensemble_all_fused_submission.zip
  top3   members=2 dev dense F1@7 0.1234 -> /kaggle/working/p7b_ensemble_top3_submission.zip
         fused (alpha=0.4) dev F1@7 0.4414 -> /kaggle/working/p7b_ensemble_top3_fused_submission.zip
